In [0]:
RAW_PATH='/Volumes/workspace/default/eccomerce/'
BRONZE_PATH='/Volumes/workspace/default/eccomerce/bronze/'
SILVER_PATH='/Volumes/workspace/default/eccomerce/silver/'
GOLD_PATH='/Volumes/workspace/default/eccomerce/gold/'

#KEY OF GCP PROJECT
GCP_PROJECT='decisive-cinema-489218-g0'
BQ_QUERY='ecommerce'
TEMP_GCS_BUCKET='ecommerce-databricks-temp'


#secrets codes
GCP_SECRET_SCOPE='gcp-secret'
GCP_SECRET_KEY='gcp-sa-key'

#Authentication

In [0]:
# Service account credentials file path (to be used with BigQuery connector)
CREDENTIALS_FILE = "/Volumes/workspace/default/eccomerce/decisive-cinema-489218-g0-58cbdcb90fde.json"

# Verify credentials file exists and is valid
import json
with open(CREDENTIALS_FILE, 'r') as f:
    creds = json.load(f)
    print(f"✓ Authenticated as: {creds.get('client_email')}")
    print(f"✓ Project: {creds.get('project_id')}")
    print(f"\nUse this path when reading/writing BigQuery: credentialsFile={CREDENTIALS_FILE}")

✓ Authenticated as: databricks-bigquery@decisive-cinema-489218-g0.iam.gserviceaccount.com
✓ Project: decisive-cinema-489218-g0

Use this path when reading/writing BigQuery: credentialsFile=/Volumes/workspace/default/eccomerce/decisive-cinema-489218-g0-58cbdcb90fde.json


In [0]:
# Note: BigQuery connector options like credentialsFile, temporaryGcsBucket, and parentProject
# should be specified as DataFrame options when reading/writing, not as global Spark configs.
df = spark.read.format("bigquery") \
     .option("credentialsFile", "/Volumes/workspace/default/eccomerce/decisive-cinema-489218-g0-58cbdcb90fde.json") \
     .option("temporaryGcsBucket", TEMP_GCS_BUCKET) \
     .option("parentProject", GCP_PROJECT) \
     .option("table", "project.dataset.table") \
     .load()


In [0]:
tables={
    'daily_store_category': GOLD_PATH +'daily_store_category',
    'top_customers': GOLD_PATH +'top_customers',
    'promo_impact': GOLD_PATH +'promo_impact',
    'product_sentiment': GOLD_PATH +'product_sentiment',
    'rfm': GOLD_PATH+'rfm'
}

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()
print('bigquery connector loaded')

bigquery connector loaded


In [0]:
for table_name, delta_path in tables.items():
    df=spark.read.format('delta').load(delta_path)
    print(f"writing {table_name} to Bigquery {GCP_PROJECT}.{BQ_QUERY}.{table_name}")
    df.write.mode("overwrite").option('header', 'true').csv('dbfs:/Volumes/workspace/default/eccomerce/{table_name}.csv')

writing daily_store_category to Bigquery decisive-cinema-489218-g0.ecommerce.daily_store_category
writing top_customers to Bigquery decisive-cinema-489218-g0.ecommerce.top_customers
writing promo_impact to Bigquery decisive-cinema-489218-g0.ecommerce.promo_impact
writing product_sentiment to Bigquery decisive-cinema-489218-g0.ecommerce.product_sentiment
writing rfm to Bigquery decisive-cinema-489218-g0.ecommerce.rfm


Read and download all files data

In [0]:
df=spark.read.format('delta').load('/Volumes/workspace/default/eccomerce/gold/daily_store_category')
df.display()

transaction_date,store_name,store_location,category,gross_sales,net_sales,unique_customer,txn_count
2025-10-02,TechWorld,Mumbai,Electronics,2375.34,1808.787,2,4
2025-10-02,TechWorld,Mumbai,Wearable,782.15,782.15,1,1
2025-10-02,StyleStop,Chennai,Electronics,396.37,396.37,1,1
2025-10-02,GadgetHub,Pune,null,659.77,659.77,2,2
2025-10-02,StyleStop,Chennai,Accessories,829.72,589.1012,1,1
2025-10-02,FreshMart,Bangalore,null,447.88,447.88,1,1
2025-10-02,TechWorld,Mumbai,null,174.29,174.29,1,1
2025-10-04,GadgetHub,Pune,Storage,733.41,733.41,1,1
2025-10-04,TechWorld,Mumbai,Storage,167.42,167.42,1,1
2025-10-04,StyleStop,Chennai,null,969.84,969.84,1,1


In [0]:
df=spark.read.format('delta').load('/Volumes/workspace/default/eccomerce/gold/top_customers')
df.display()

customer_id,customer_name,store_name,region,total_spent,txn_count
9,Vikas Nair,TechWorld,East,1528.4299999999998,3
9,Vikas Nair,GadgetHub,East,1396.096,2
8,Neha Singh,StyleStop,North,1232.2812,2
11,Sanjay Kulkarni,StyleStop,West,969.84,1
5,Arjun Mehta,TechWorld,North,924.432,2
10,Pooja Das,TechWorld,South,884.355,2
12,Deepa Menon,TechWorld,North,782.15,1
14,Anjali Joshi,GadgetHub,East,733.41,1
9,Vikas Nair,BookNest,East,678.192,2
13,Manish Gupta,StyleStop,South,572.25,1


In [0]:
df=spark.read.format('delta').load('/Volumes/workspace/default/eccomerce/gold/promo_impact')
df.display()

promotion_id,promo_txns,promo_sales,avg_with_promo
5,2,1087.7475,543.87375
2,2,1232.7804999999998,616.3902499999999
9,1,420.83099999999996,420.83099999999996
6,3,1267.2931999999998,422.4310666666666
8,1,463.524,463.524
10,1,253.61280000000002,253.61280000000002
7,1,222.76800000000003,222.76800000000003
3,1,738.834,738.834
4,1,622.176,622.176


In [0]:
df=spark.read.format('delta').load('/Volumes/workspace/default/eccomerce/gold/product_sentiment')
df.display()

product_id,product_name,category,avg_rating,rating_count
11,null,null,2.0,1


In [0]:
df=spark.read.format('delta').load('/Volumes/workspace/default/eccomerce/gold/rfm')
df.display()

customer_id,customer_name,last_txn,frequency,monetary,recency_days,recency_bucket,frequency_bucket,monetary_bucket
12,Deepa Menon,2025-10-03,3,1258.5308,155,90+,medium,high
10,Pooja Das,2025-10-02,4,1715.8249999999998,156,90+,medium,high
11,Sanjay Kulkarni,2025-10-04,2,1246.02,154,90+,low,high
5,Arjun Mehta,2025-10-02,2,924.432,156,90+,low,medium
15,Rohit Khanna,2025-10-02,1,396.37,156,90+,low,medium
13,Manish Gupta,2025-10-03,2,746.54,155,90+,low,medium
8,Neha Singh,2025-10-04,2,1232.2812,154,90+,low,high
9,Vikas Nair,2025-10-04,7,3602.718,154,90+,medium,high
14,Anjali Joshi,2025-10-04,1,733.41,154,90+,low,medium
7,Rahul Verma,2025-10-01,1,59.88,157,90+,low,low


 **Steps to upload files to BigQuery**
1. Download all the 5 files
2. Go to big query option
3. Search for bigquery on gcp
4. Click on create dataset
5. Provide dataset id(project id)
6. Go to dataset
7. Click on create table
8. Give name of table -rfm (do the same for all 5 tables)

**Steps to connect with Looker studio**
1. Open Looker Studio
2. Create a New Report
3. Add Data Source
4. Select BigQuery Connector
5. click on my project (it will automatically detect my project)
6. click on project id , then click on dataset, then we will able to see all the tables
7. select daily_store_category table and click on add option ( so we are adding this daily_store_category query and analysing it with BI tool   looker studio)
8. it will craete chart for table which is faster interactive dashboard which allow us to generate reports based on BIGQUERY

Automation of data---pipeline
